# ChibiCreate — FASE 3B via Google Colab

Prova real: **Colab → ComfyUI → Qwen-Image-Edit-2511 → waifu_001 → output**.
Duas execuções com os mesmos parâmetros.

---

## Leia antes de rodar

**Colab é infraestrutura EXPERIMENTAL e TEMPORÁRIA.** Não é a arquitetura do
projeto. Nada aqui altera o repositório além de gravar dois diretórios em
`experiments/`. A arquitetura permanece: `LOCAL → CLI → COMFYUI REMOTO → GPU
NVIDIA`. O Colab está apenas ocupando, por uma sessão, a caixa "GPU NVIDIA".

**Nenhum código novo de cliente.** Este notebook chama a CLI que já existe
(`chibi comfy preflight`, `chibi experiment qwen-edit`). Se algo falhar, o
conserto é no repositório, não em código solto de notebook.

### Requisito de VRAM — leia com atenção

| Variante 2511 | Pesos | Cabe na T4 (15 GB)? |
|---|---|---|
| `bf16` | 40.9 GB | não |
| `fp8mixed` | **20.5 GB** | **não** |
| `int8_convrot` | 20.5 GB | não |

**O Colab gratuito entrega uma T4 de 16 GB (≈15 GB utilizáveis).** O modelo
tem 20.5 GB só de pesos. Na prática isso significa:

> **T4 não serve para este experimento.** A célula 1 vai detectar e PARAR com
> `COLAB_GPU_INSUFFICIENT`. Isso é o comportamento correto, não um bug.

É necessário **Colab Pro (L4, 22.5 GB)** ou **Pro+ (A100, 40 GB)**. O
ComfyUI faz offload de blocos para a RAM, então a L4 tende a funcionar com
o encoder sendo descarregado entre etapas — mas isso é `[TEST REQUIRED]`,
não promessa.

### Runtime

`Ambiente de execução → Alterar o tipo de ambiente de execução → GPU`.
Se tiver Pro, escolha **L4** ou **A100**, não T4.


---

## Célula 1 — detectar GPU (portão de parada)

Não assume T4, L4 nem A100. Detecta, registra e decide.


In [ ]:
import subprocess, sys, json, os

# Pesos da variante escolhida (fp8mixed). Fonte: models.lock.yaml.
WEIGHTS_GB = 20.53
# Margem para ativacoes, VAE e latentes. Estimativa, nao medicao.
MIN_VRAM_GB = 22.0

try:
    import torch
except ImportError:
    torch = None

print('=' * 62)
print('AMBIENTE COLAB — detectado, nao presumido')
print('=' * 62)
print('Python :', sys.version.split()[0])
print('Torch  :', torch.__version__ if torch else 'ausente')

if torch is None or not torch.cuda.is_available():
    print('CUDA   : INDISPONIVEL')
    print()
    print('COLAB_GPU_INSUFFICIENT — nenhuma GPU CUDA.')
    print('Runtime -> Alterar tipo de ambiente de execucao -> GPU')
    raise SystemExit('FASE 3B permanece BLOCKED')

props = torch.cuda.get_device_properties(0)
total_gb = props.total_memory / 1024 ** 3
free_b, _ = torch.cuda.mem_get_info()
free_gb = free_b / 1024 ** 3

GPU_INFO = {
    'name': props.name,
    'vram_total_gb': round(total_gb, 2),
    'vram_free_gb': round(free_gb, 2),
    'cuda': torch.version.cuda,
    'capability': f'{props.major}.{props.minor}',
    'torch': torch.__version__,
    'python': sys.version.split()[0],
}
print('GPU    :', GPU_INFO['name'])
print('VRAM   : %.2f GB total / %.2f GB livre' % (total_gb, free_gb))
print('CUDA   :', GPU_INFO['cuda'], '| capability', GPU_INFO['capability'])
print()

# bf16 exige Ampere+ (capability >= 8.0). T4 (7.5) nao tem.
GPU_INFO['bf16_supported'] = props.major >= 8
if not GPU_INFO['bf16_supported']:
    print('AVISO: esta GPU nao suporta bfloat16 nativo (capability < 8.0).')

print('Necessario : %.1f GB (pesos %.1f GB + margem)' % (MIN_VRAM_GB, WEIGHTS_GB))

if total_gb < MIN_VRAM_GB:
    print()
    print('=' * 62)
    print('COLAB_GPU_INSUFFICIENT')
    print('=' * 62)
    print('{} tem {:.1f} GB; o Qwen-Image-Edit-2511 precisa de ~{:.0f} GB.'
          .format(GPU_INFO['name'], total_gb, MIN_VRAM_GB))
    print()
    print('PARE AQUI. Nao fazer inferencia em CPU, nao trocar de modelo,')
    print('nao usar quantizacao comunitaria sem aprovacao humana.')
    print()
    print('Use Colab Pro (L4 22.5 GB) ou Pro+ (A100 40 GB).')
    print('Registre este resultado e mantenha FASE 3B = BLOCKED.')
    raise SystemExit('COLAB_GPU_INSUFFICIENT')

print()
print('GPU ADEQUADA — pode prosseguir.')
json.dump(GPU_INFO, open('/content/gpu_info.json', 'w'), indent=2)


---

## Célula 2 — clonar o repositório

Traz a CLI, o workflow, os configs e a arte de referência. **Nada é
reimplementado aqui.**


In [ ]:
%cd /content
!git clone --branch arena/01a07ece-chibicreate \
    https://github.com/BloomRX/ChibiCreate.git 2>/dev/null || echo 'ja clonado'
%cd /content/ChibiCreate
!git log --oneline -1
!pip install -q pyyaml pillow numpy

import hashlib, pathlib
ref = pathlib.Path('characters/waifu_001/reference/full_body.png')
if not ref.exists():
    print('reference ausente — regenerando com o Flow 01 (deterministico)')
    !python -m scripts.chibi.cli flow01 waifu_001 --force

h = hashlib.sha256(ref.read_bytes()).hexdigest()
print('input      :', ref)
print('sha256     :', h)
EXPECTED = '2fdcd5f428f5980d63e31d4bf4a67aecbc11c1b101c19ca75f819db616cb8177'
print('confere    :', 'SIM' if h == EXPECTED else 'NAO — investigar antes de seguir')


---

## Célula 3 — instalar o ComfyUI oficial

ComfyUI oficial (`comfyanonymous/ComfyUI`), sem custom nodes. O workflow
usa apenas nodes nativos.


In [ ]:
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || echo 'ja clonado'
%cd /content/ComfyUI
!git log --oneline -1
!pip install -q -r requirements.txt

import subprocess
COMFY_COMMIT = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd='/content/ComfyUI').decode().strip()
print('ComfyUI commit:', COMFY_COMMIT)


---

## Célula 4 — baixar os modelos

**Somente os 3 arquivos do workflow.** Todos de repositórios oficiais
**Comfy-Org**, Apache-2.0 — não são quantizações comunitárias arbitrárias.

| Arquivo | Origem | Tamanho | Licença |
|---|---|---|---|
| `qwen_image_edit_2511_fp8mixed` | `Comfy-Org/Qwen-Image-Edit_ComfyUI` | 20.5 GB | Apache-2.0 |
| `qwen_2.5_vl_7b_fp8_scaled` | `Comfy-Org/Qwen-Image_ComfyUI` | 9.4 GB | Apache-2.0 |
| `qwen_image_vae` | `Comfy-Org/Qwen-Image_ComfyUI` | 0.25 GB | Apache-2.0 |

~30 GB no total. Os SHA256 esperados estão em `config/models.lock.yaml` e
são conferidos após o download.

> **Se faltar disco:** o Colab dá ~78 GB no disco da GPU. Cabe. Não tente
> contornar com quantização de terceiros — pare e reporte.


In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
import pathlib, shutil

M = pathlib.Path('/content/ComfyUI/models')

DOWNLOADS = [
    ('Comfy-Org/Qwen-Image-Edit_ComfyUI',
     '984166f60a9b1fcede5e9b9287b7a7aebc050010',
     'split_files/diffusion_models/qwen_image_edit_2511_fp8mixed.safetensors',
     M / 'diffusion_models',
     'c9fdc158e46d3b61ef75f21ae866ca2fe808bf4a53643120d1c1e87c19280a4e'),
    ('Comfy-Org/Qwen-Image_ComfyUI',
     '7beb7b647f04469fbe64ba8adc2bb0d7e5e9f73f',
     'split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors',
     M / 'text_encoders',
     'cb5636d852a0ea6a9075ab1bef496c0db7aef13c02350571e388aea959c5c0b4'),
    ('Comfy-Org/Qwen-Image_ComfyUI',
     '7beb7b647f04469fbe64ba8adc2bb0d7e5e9f73f',
     'split_files/vae/qwen_image_vae.safetensors',
     M / 'vae',
     'a70580f0213e67967ee9c95f05bb400e8fb08307e017a924bf3441223e023d1f'),
]

MODEL_RECORD = []
for repo, rev, remote, dest, expected_sha in DOWNLOADS:
    dest.mkdir(parents=True, exist_ok=True)
    fname = remote.split('/')[-1]
    target = dest / fname
    if target.exists():
        print('ja existe:', fname)
    else:
        print('baixando :', fname)
        got = hf_hub_download(repo_id=repo, revision=rev, filename=remote)
        shutil.copy(got, target)
    MODEL_RECORD.append({
        'file': fname, 'repo': repo, 'revision': rev,
        'license': 'Apache-2.0',
        'size_bytes': target.stat().st_size,
        'sha256_expected': expected_sha,
    })

print()
!df -h /content | tail -1


## Célula 4b — conferir SHA256

Confere o hash real contra `models.lock.yaml`. Leva alguns minutos (30 GB).
Divergência = **pare**: o arquivo não é o que dizemos que é.


In [ ]:
import hashlib, pathlib, json

def sha256_of(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

ok = True
for rec in MODEL_RECORD:
    hits = list(pathlib.Path('/content/ComfyUI/models').rglob(rec['file']))
    actual = sha256_of(hits[0])
    rec['sha256_actual'] = actual
    match = actual == rec['sha256_expected']
    rec['sha256_match'] = match
    ok &= match
    print(('OK   ' if match else 'FALHA'), rec['file'])
    if not match:
        print('   esperado:', rec['sha256_expected'])
        print('   obtido  :', actual)

json.dump(MODEL_RECORD, open('/content/model_record.json', 'w'), indent=2)
if not ok:
    raise SystemExit('SHA256 divergente — PARE e reporte.')
print()
print('Todos os pesos conferem com models.lock.yaml.')


---

## Célula 5 — iniciar o ComfyUI

Em background, na porta 8188. Como a CLI roda **no mesmo Colab**, não é
preciso túnel externo — `127.0.0.1:8188` basta. Menos peça móvel, menos
exposição.

> Um túnel (cloudflared) só seria necessário para acessar a UI do navegador.
> Isso é opcional e está na célula 10.


In [ ]:
import subprocess, time, urllib.request, json

LOG = open('/content/comfyui.log', 'w')
proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188'],
    cwd='/content/ComfyUI', stdout=LOG, stderr=subprocess.STDOUT)

print('subindo ComfyUI (carregar 20 GB leva alguns minutos)...')
base = 'http://127.0.0.1:8188'
for i in range(180):
    time.sleep(5)
    try:
        with urllib.request.urlopen(base + '/system_stats', timeout=5) as r:
            stats = json.load(r)
        print('no ar apos ~%ds' % ((i + 1) * 5))
        break
    except Exception:
        if proc.poll() is not None:
            print(open('/content/comfyui.log').read()[-3000:])
            raise SystemExit('ComfyUI morreu ao iniciar')
else:
    print(open('/content/comfyui.log').read()[-3000:])
    raise SystemExit('ComfyUI nao respondeu em 15 min')

print(json.dumps(stats.get('system', {}), indent=2))
for d in stats.get('devices', []):
    print('device:', d.get('name'), '| vram_total', d.get('vram_total'))


---

## Célula 6 — FASE A: preflight

Usa a CLI do repositório — a mesma que rodaria contra qualquer GPU.

**Portão.** Só siga se disser `PRONTO para uma execução real`.
Se der `WORKFLOW_INCOMPATIBLE`, ele imprime `node/expected/actual`:
**copie essas linhas**, é o dado exato para corrigir o workflow.


In [ ]:
import os
os.environ['CHIBI_COMFY_URL'] = 'http://127.0.0.1:8188'
%cd /content/ChibiCreate

!python -m scripts.chibi.cli comfy status    --env cloud
print('=' * 62)
!python -m scripts.chibi.cli comfy preflight --env cloud
print('=' * 62)
!python -m scripts.chibi.cli comfy validate  --env cloud


---

## Célula 7 — FASE B: primeira execução real

Prompt, seed e batch exatamente como especificado. Resultado é
**EXPERIMENTAL** — vai para `experiments/`, nunca para
`characters/waifu_001/chibi/master.png`.


In [ ]:
PROMPT = ('Transform this character into a clean stylized chibi full-body '
          'character, preserving the same identity, black hair, red eyes, '
          'horns, black outfit, long black cape and golden ornaments.')

%cd /content/ChibiCreate
!python -m scripts.chibi.cli experiment qwen-edit \
    --character waifu_001 \
    --input reference/full_body.png \
    --seed 42 \
    --env cloud \
    --prompt "$PROMPT"


## Célula 8 — FASE C: validar a primeira execução


In [ ]:
import pathlib, json
from PIL import Image

runs = sorted(pathlib.Path('experiments').rglob('run_*'))
print('execucoes:', [str(r) for r in runs])
run1 = runs[0]

for f in sorted(run1.iterdir()):
    print('  %-28s %10d bytes' % (f.name, f.stat().st_size))

img = Image.open(run1 / 'output.png')
print()
print('output:', img.size, img.mode)
display(img)

recipe = json.load(open(run1 / 'recipe.json'))
for k in ('artifact_sha256', 'input_hashes', 'workflow_sha256', 'seed',
          'steps', 'cfg', 'sampler', 'width', 'height', 'gpu', 'cuda',
          'execution_time', 'approval_status'):
    print('%-18s %s' % (k, recipe.get(k)))


---

## Célula 9 — FASE E: segunda execução + comparação

Mesmo input, prompt, seed, parâmetros, workflow, revision e GPU.

> Hash diferente entre as duas é **resultado válido**. Medimos a variação;
> nunca prometemos determinismo.


In [ ]:
%cd /content/ChibiCreate
!python -m scripts.chibi.cli experiment qwen-edit \
    --character waifu_001 \
    --input reference/full_body.png \
    --seed 42 \
    --env cloud \
    --prompt "$PROMPT"


In [ ]:
import pathlib
runs = sorted(pathlib.Path('experiments').rglob('run_*'))
a, b = runs[0], runs[1]
!python -m scripts.chibi.cli experiment compare {a} {b}

from PIL import Image
print('run_001'); display(Image.open(a / 'output.png'))
print('run_002'); display(Image.open(b / 'output.png'))


---

## Célula 10 — registrar custo e ambiente Colab


In [ ]:
import json, pathlib, subprocess

colab_meta = {
    'infrastructure': 'google_colab',
    'infrastructure_status': 'EXPERIMENTAL_TEMPORARY',
    'gpu': json.load(open('/content/gpu_info.json')),
    'models': json.load(open('/content/model_record.json')),
    'comfyui_commit': COMFY_COMMIT,
    'cost': 0,
    'cost_note': 'No direct GPU cost observed in this session.',
    'cost_warning': ('Nao extrapolar para custo de producao. Colab nao expoe '
                     'custo por execucao; tiers pagos consomem compute units '
                     'nao contabilizadas aqui.'),
    'limitations': [
        'Sessao efemera: tudo em /content e perdido ao desconectar.',
        'GPU nao garantida: pode variar entre sessoes.',
        'Desconexao por inatividade interrompe execucoes longas.',
        'Colab NAO e infraestrutura permanente do projeto.',
    ],
}

out = pathlib.Path('/content/ChibiCreate/experiments/colab_session.json')
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(colab_meta, indent=2))
print(json.dumps(colab_meta, indent=2))


---

## Célula 11 — salvar os resultados

**A sessão do Colab é efêmera.** Sem isto, os outputs somem.

O push exige um token com permissão de escrita. Se preferir não colar token
no Colab — o que é razoável — use a opção de download.


In [ ]:
# Opcao A — baixar como zip (sem credencial)
import shutil
shutil.make_archive('/content/fase3b_resultados', 'zip',
                    '/content/ChibiCreate/experiments')
from google.colab import files
files.download('/content/fase3b_resultados.zip')


In [ ]:
# Opcao B — push direto (requer token com escrita)
# Descomente e preencha. O token NAO deve ser commitado nem ficar no notebook salvo.
#
# from getpass import getpass
# TOKEN = getpass('GitHub token: ')
# %cd /content/ChibiCreate
# !git config user.email 'colab@chibicreate.local'
# !git config user.name  'ChibiCreate Colab'
# !git add -f experiments/
# !git commit -m 'FASE 3B: duas execucoes reais via Colab (EXPERIMENTAL)'
# !git push https://$TOKEN@github.com/BloomRX/ChibiCreate.git arena/01a07ece-chibicreate


---

## PARE AQUI

O experimento termina com **duas execuções reais + metadata**.

Não seguir para: Flow 02, 8 candidatos, `master.png`, ControlNet, LoRA
(Style ou Character), Flow 03 ou animação.

### [HUMAN REVIEW REQUIRED]

A decisão sobre qualidade, identidade, necessidade de LoRA e ajuste de
prompt é **humana**. Olhe os dois outputs e avalie:

- [ ] cabelo preto
- [ ] olhos vermelhos
- [ ] chifres
- [ ] roupa preta
- [ ] manto
- [ ] ornamentos dourados
- [ ] silhueta
- [ ] a personagem continua reconhecível

### FASE 3B só vira COMPLETE

depois de **duas execuções reais concluídas** — não por este notebook
existir. Enquanto isso: `BLOCKED`.
